In [ ]:
# ── Source-of-truth note ──────────────────────────────────────────────────
#
# Earlier versions of this notebook held a string copy of src/ranker.py /
# src/report_generator.py in this cell and wrote it back to disk on every run.
# That pattern was a footgun: every "Run All" silently reverted the source file
# to whatever snapshot was frozen here, so improvements made directly in src/
# would disappear (e.g. Finding.to_dict, profile re-ranking, the numeric
# grounding validator, prompt caching).
#
# The notebook now treats src/ as the single source of truth. The cells below
# IMPORT from src/, they do not generate or overwrite it.
print("Using src/ranker.py and src/report_generator.py as the source of truth.")


In [ ]:
import importlib, sys
from pathlib import Path
from typing import Union, Optional

# ── Add src to path ───────────────────────────────────────────────────────────
sys.path.insert(0, str(Path("..").resolve() / "src"))

# ── Fresh import (clears any cached version) ──────────────────────────────────
import ranker
importlib.reload(ranker)
from ranker import MetricRanker
import pandas as pd

# ── Find project root ─────────────────────────────────────────────────────────
def find_project_root(start):
    for p in [start, *start.parents]:
        if (p / "data" / "metrics_159d.csv").exists():
            return p
    raise FileNotFoundError(
        f"Cannot find data/metrics_159d.csv anywhere above {start}\n"
        f"Make sure metrics_159d.csv is inside the data/ folder."
    )

ROOT = find_project_root(Path.cwd())
CSV  = ROOT / "data" / "metrics_159d.csv"

print(f"Project root : {ROOT}")
print(f"CSV path     : {CSV}")
print(f"CSV exists   : {CSV.exists()}")
print("Import successful")

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
# This builds all series and pre-computes rolling stats.
# Expect it to take 10-30 seconds on 160 days of data.

ranker_obj = MetricRanker(CSV)

first, last = ranker_obj.date_range()
all_dates   = ranker_obj.available_dates()

print(f"\nAvailable date range : {first}  →  {last}")
print(f"Total dates          : {len(all_dates)}")

# Pick a test date 30 days before the end so we have plenty of history
test_date = all_dates[-30]
print(f"\nTest date selected   : {test_date}")
print(f"(30th from last — enough history, recent enough to be interesting)")

In [ ]:
# ── Score the test date ───────────────────────────────────────────────────────
findings = ranker_obj.rank_day(test_date, top_n=10)

print(f"Top 10 findings for {test_date}")
print("=" * 80)

for i, f in enumerate(findings, 1):
    alert_tag = "  *** ALERT ***" if f.is_alert else ""
    print(f"\n#{i}{alert_tag}")
    print(f"  {f.fact_sentence}")
    print(f"  score={f.final_score:.4f}  "
          f"(stat={f.stat_score:.3f} x biz_weight={f.biz_weight})")

print(f"\n{'='*80}")
print(f"Total findings scored : {len(findings)}")
print(f"Alerts (|z|>2)        : {sum(1 for f in findings if f.is_alert)}")

In [ ]:
# ── Weekly ranking ────────────────────────────────────────────────────────────
# Use the last date in the dataset as the week-end date
week_end      = all_dates[-1]
week_findings = ranker_obj.rank_week(week_end, top_n=20)

print(f"Top 20 weekly findings — week ending {week_end}")
print("=" * 80)

for i, f in enumerate(week_findings, 1):
    alert_tag = "***" if f.is_alert else "   "
    print(f"\n#{i} {alert_tag} [{f.date}]")
    print(f"     {f.fact_sentence}")
    print(f"     final_score={f.final_score:.4f}")

print(f"\n{'='*80}")
print(f"Total weekly findings : {len(week_findings)}")
print(f"Alerts                : {sum(1 for f in week_findings if f.is_alert)}")

In [ ]:
# ── Validate scoring logic on the top finding ─────────────────────────────────
# This is the most important cell — it shows every number and lets you
# verify the math is working correctly before trusting the outputs.

top = findings[0]

print("Score breakdown for top finding")
print("=" * 60)
print(f"Metric         : {top.metric}")
print(f"Source         : {top.source}")
print(f"Channel        : {top.channel if top.channel else '(total)'}")
print(f"Campaign       : {top.campaign if top.campaign else '(none)'}")
print(f"Customer type  : {top.customer_type if top.customer_type else '(all)'}")
print()
print(f"Value today    : {top.value:>15,.2f}")
print(f"Baseline (DoW) : {top.baseline_mean:>15,.2f}")
print(f"Baseline std   : {top.baseline_std:>15,.2f}")
print()

z_str = f"{top.z_score:+.3f}"
alert_str = "  <-- ALERT (|z| > 2.0)" if top.is_alert else ""
print(f"z-score        : {z_str:>15}{alert_str}")

if top.wow_delta_pct is not None:
    print(f"WoW delta      : {top.wow_delta_pct:>+14.1f}%")
else:
    print(f"WoW delta      :     (not available)")

if top.mom_delta_pct is not None:
    print(f"MoM delta      : {top.mom_delta_pct:>+14.1f}%")
else:
    print(f"MoM delta      :     (not available)")

print()
print(f"stat_score     : {top.stat_score:>15.4f}")
print(f"  formula      : |z|/3 x 0.4  +  |wow|/100 x 0.4  +  |mom|/100 x 0.2")

# Show the actual arithmetic
z_component   = min(abs(top.z_score) / 3, 1.0) * 0.4
wow_component = min(abs(top.wow_delta_pct or 0) / 100, 1.0) * 0.4
mom_component = min(abs(top.mom_delta_pct or 0) / 100, 1.0) * 0.2
print(f"  breakdown    : {z_component:.4f}  +  {wow_component:.4f}  +  {mom_component:.4f}")

print()
print(f"biz_weight     : {top.biz_weight:>15}  (metric: {top.metric})")
print(f"final_score    : {top.final_score:>15.4f}  =  stat_score x biz_weight")
print()
print(f"Direction      : {top.direction}")
print(f"Is alert       : {top.is_alert}")
print()
print(f"Fact sentence  :")
print(f"  {top.fact_sentence}")

In [ ]:
# ── Score multiple dates — these become Phase 3 sample outputs ────────────────
# 3 daily digests + 2 weekly reports, using real data from your dataset

print("DAILY DIGEST CANDIDATES")
print("=" * 80)

sample_daily_dates = [
    "2025-12-10",   # known anomaly from EDA (high revenue spike)
    all_dates[-10], # recent normal day
    all_dates[-45], # mid-dataset day
]

for date in sample_daily_dates:
    day_findings = ranker_obj.rank_day(date, top_n=10)
    alerts = sum(1 for f in day_findings if f.is_alert)
    print(f"\nDate: {date}  |  alerts: {alerts}  |  top score: {day_findings[0].final_score:.4f}")
    for f in day_findings[:5]:
        marker = "***" if f.is_alert else "   "
        print(f"  {marker} {f.fact_sentence}")

print()
print("WEEKLY REPORT CANDIDATES")
print("=" * 80)

sample_weekly_dates = [
    all_dates[-7],
    all_dates[-1],
]

for week_end in sample_weekly_dates:
    wk_findings = ranker_obj.rank_week(week_end, top_n=20)
    alerts = sum(1 for f in wk_findings if f.is_alert)
    print(f"\nWeek ending: {week_end}  |  alerts: {alerts}")
    for f in wk_findings[:6]:
        marker = "***" if f.is_alert else "   "
        print(f"  {marker} [{f.date}] {f.fact_sentence}")

# ── Save findings as structured data for Phase 3 ──────────────────────────────
import json

def steady_findings_for_date(ranker, date, k=3):
    """
    Find low-scoring (i.e. stable) findings on `date` to ground the
    "What's holding steady" section. Restricted to high-business-weight
    metrics so we don't reassure users about impressions being steady.
    """
    all_today = ranker.rank_day(date, top_n=200)
    stable_metrics = {"revenue", "orders", "aov", "roas", "spend", "new_customer_share"}
    steady = [
        f for f in all_today
        if not f.is_alert
        and not f.is_data_quality_flag
        and f.metric in stable_metrics
        and abs(f.z_score) < 1.0
    ]
    # Sort by lowest |z| first — most stable
    steady.sort(key=lambda f: abs(f.z_score))
    return steady[:k]

# Use Finding.to_dict() so is_data_quality_flag is preserved end-to-end.
output = {
    "daily": {
        date: [f.to_dict() for f in ranker_obj.rank_day(date, top_n=10)]
        for date in sample_daily_dates
    },
    "daily_steady": {
        date: [f.to_dict() for f in steady_findings_for_date(ranker_obj, date, k=3)]
        for date in sample_daily_dates
    },
    "weekly": {
        week_end: [f.to_dict() for f in ranker_obj.rank_week(week_end, top_n=20)]
        for week_end in sample_weekly_dates
    },
    "weekly_steady": {
        week_end: [f.to_dict() for f in steady_findings_for_date(ranker_obj, week_end, k=3)]
        for week_end in sample_weekly_dates
    },
}

OUTPUT_PATH = ROOT / "outputs"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_PATH / "ranked_findings.json", "w", encoding="utf-8") as fh:
    json.dump(output, fh, indent=2, ensure_ascii=False)

print(f"\nSaved ranked_findings.json to {OUTPUT_PATH}")
print("Phase 2 complete. Ready for Phase 3.")
